# Task 4 output profile

Checks that every variant row in `output/*.csv` is unique, and lists any value that repeats.

In [ ]:
import csv, glob
from collections import Counter

data = {}
for path in sorted(glob.glob("output/*.csv")):
    with open(path, encoding="utf-8") as f:
        data[path] = list(csv.DictReader(f))
list(data)

In [ ]:
keys = {
    "variant_id": lambda r: r["variant_id"],
    "url": lambda r: r["url"],
    "product + variant name": lambda r: (r["product_name"], r["variant_name"]),
}
for path, rows in data.items():
    print(f"{path}: {len(rows)} rows, {len({r['product_id'] for r in rows})} products")
    for label, key in keys.items():
        counts = Counter(key(r) for r in rows)
        repeated = {k: c for k, c in counts.items() if c > 1}
        status = "all unique" if not repeated else f"{len(repeated)} repeated"
        print(f"   {label:24} {len(counts):>4} distinct  -> {status}")
        for k, c in list(repeated.items())[:10]:
            print(f"      x{c}: {k}")

In [ ]:
# Empty values per column.
for path, rows in data.items():
    empty = {col: sum(not r[col] for r in rows) for col in rows[0]}
    print(path, {col: n for col, n in empty.items() if n})

In [ ]:
# Products with the most variants, and how many variants are out of stock.
for path, rows in data.items():
    per_product = Counter(r["product_name"] for r in rows)
    out = sum(r["in_stock"] == "False" for r in rows)
    print(f"{path}: {out} variants out of stock | most variants: {per_product.most_common(5)}")